# KAN-CDSCO2004U  Machine Learning and Deep Learning

## Lab 15: Image Classification with CNNs — ✅ Solution
**Estimated time: 1.5 hours**

### Learning Objectives
By the end of this exercise, you will be able to:
- Download and preprocess an **image dataset** using `ImageDataGenerator`
- Build a **Convolutional Neural Network (CNN)** with Keras
- **Train and evaluate** a CNN for binary image classification
- **Visualize** training vs validation accuracy and loss curves

**How to work through this notebook:**
- 🏃 **RUN** cells = Just execute the code to see the output
- ✏️ **TODO** cells = Write your own code or answer questions
- 📖 **READ** cells = Explanations to help you understand the concepts

## <center> Lecture-15 "Image Classification Using Convolutional Neural Networks"

📖 **READ**

# Classify Cats vs Dogs from Images Using a CNN

In this lab you will build an image classifier that decides whether a photo shows a **cat** or a **dog**.

**Task overview:**
- Build a CNN model consisting of **three convolutional blocks** (each followed by a MaxPooling layer), a **Flatten** layer, a **Dense(512, relu)** layer, and a final **Dense(1)** output layer.
- Use the **Adam** optimizer and **binary cross-entropy** loss.
- Train with `batch_size = 128`, `epochs = 15`, `IMG_HEIGHT = 150`, `IMG_WIDTH = 150`.

**Hints:**
- Use `tf.keras.Sequential` to stack layers.
- Use `tf.keras.preprocessing.image.ImageDataGenerator` to load and preprocess the images from disk.

---
# Part 1 — Data Preprocessing and Exploration
---

🏃 **RUN**

## Step 1: Import Libraries

Run this cell to import all the Python packages you will need throughout the exercise.

In [ ]:
# 🏃 RUN — Import packages
# Author: Luca Gudi (lgg.digi@cbs.dk)
# Src: TensorFlow.org
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `import os` | Provides functions to interact with the operating system, used here to build file paths. |
| `import numpy as np` | Imports NumPy for numerical array operations. |
| `import matplotlib.pyplot as plt` | Imports Matplotlib's plotting interface to visualise images and training metrics. |
| `import tensorflow as tf` | Imports TensorFlow, the deep learning framework used throughout this lab. |
| `from tensorflow.keras.models import Sequential` | Imports the `Sequential` container that lets us stack layers one after another. |
| `from tensorflow.keras.layers import ...` | Imports the individual layer types (`Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`) needed to build the CNN. |
| `from tensorflow.keras.preprocessing.image import ImageDataGenerator` | Imports the utility class that reads images from disk, applies rescaling, and streams batches to the model. |

🏃 **RUN**

## Step 2: Download and Extract the Dataset

The cell below downloads the **Cats vs Dogs** dataset via `tensorflow_datasets` and saves the images into a local folder structure (`train/cats`, `train/dogs`, `validation/cats`, `validation/dogs`).

In [ ]:
# 🏃 RUN — Download the dataset
# The original Google zip link is broken, so we grab the images via
# tensorflow_datasets and save them into the folder layout the notebook expects.

import tensorflow_datasets as tfds
from PIL import Image

PATH = os.path.join(os.path.expanduser("~"), ".keras", "datasets", "cats_and_dogs_filtered")

for split in ("train", "validation"):
    for label in ("cats", "dogs"):
        os.makedirs(os.path.join(PATH, split, label), exist_ok=True)

if os.listdir(os.path.join(PATH, "train", "cats")):
    print("Dataset already exists at:", PATH)
else:
    print("Downloading cats_vs_dogs via tensorflow_datasets …")
    ds = tfds.load("cats_vs_dogs", split="train", as_supervised=True)

    LIMITS   = {"train": 1000, "validation": 500}          # images per class
    NAMES    = {0: "cats", 1: "dogs"}
    counters = {s: {0: 0, 1: 0} for s in LIMITS}

    for img, lbl in ds:
        lbl   = int(lbl)
        split = "train" if counters["train"][lbl] < LIMITS["train"] else "validation"
        if counters[split][lbl] >= LIMITS[split]:
            continue

        idx = counters[split][lbl]
        out = os.path.join(PATH, split, NAMES[lbl], f"{NAMES[lbl][:-1]}_{idx:04d}.jpg")
        Image.fromarray(img.numpy()).save(out)
        counters[split][lbl] += 1

        if all(counters[s][l] >= LIMITS[s] for s in LIMITS for l in (0, 1)):
            break

    for s in LIMITS:
        print(f"  {s:>10}: {counters[s][0]} cats, {counters[s][1]} dogs")

print("Dataset directory:", PATH)


### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `tfds.load("cats_vs_dogs", ...)` | Downloads the full Cats vs Dogs dataset via `tensorflow_datasets`. Returns a `tf.data.Dataset` of (image, label) pairs. |
| `as_supervised=True` | Returns each example as a simple `(image, label)` tuple instead of a dictionary. |
| `os.makedirs(..., exist_ok=True)` | Creates the target folder (and any missing parents). Does nothing if it already exists. |
| `Image.fromarray(img.numpy()).save(out)` | Converts a NumPy array to a PIL image and writes it to disk as JPEG. |
| `LIMITS` / `counters` | Control how many images go into each split (1 000 train + 500 validation per class). |
| `os.path.join(...)` | Builds a cross-platform file path by correctly combining directory segments. |

In [ ]:
# 🏃 RUN — Alternative: download via the original zip (currently broken)
#
# _URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
# path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=True)
# PATH = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')

✏️ **TODO**

## Step 3: Set Up Data Directories

**Objective:** Define file paths for the training and validation sets.

**Instructions:**
1. Use `os.path.join(PATH, 'train')` to create `train_dir`.
2. Use `os.path.join(PATH, 'validation')` to create `validation_dir`.
3. Define paths for `train_cats_dir`, `train_dogs_dir`, `validation_cats_dir`, `validation_dogs_dir`.
4. Print all paths to confirm they are correct.

In [ ]:
# ✏️ TODO — Define training and validation directory paths
train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')
print("Train dir:", train_dir)
print("Validation dir:", validation_dir)

In [ ]:
# ✏️ TODO — Define subdirectory paths for cats and dogs (train and validation)
train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')
print("Train cats:", train_cats_dir)
print("Train dogs:", train_dogs_dir)
print("Validation cats:", validation_cats_dir)
print("Validation dogs:", validation_dogs_dir)

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `os.path.join(PATH, 'train')` | Safely appends `'train'` to the base dataset path using the OS-specific separator (`/` on Mac/Linux, `\\` on Windows). |
| `train_dir` / `validation_dir` | Variables holding the root directories of the training and validation splits. |
| `train_cats_dir` | Path to the subfolder containing training images of cats. |
| `train_dogs_dir` | Path to the subfolder containing training images of dogs. |
| `validation_cats_dir` / `validation_dogs_dir` | Equivalent paths for validation images. |

✏️ **TODO**

## Step 4: Count Images and Set Hyperparameters

**Objective:** Understand how many images are in the dataset and define training parameters.

**Instructions:**
1. Use `len(os.listdir(...))` to count images in each subdirectory.
2. Compute `total_train` and `total_val`.
3. Print the counts.
4. Set `batch_size = 128`, `epochs = 15`, `IMG_HEIGHT = 150`, `IMG_WIDTH = 150`.

In [ ]:
# ✏️ TODO — Count training and validation images
num_cats_tr  = len(os.listdir(train_cats_dir))
num_dogs_tr  = len(os.listdir(train_dogs_dir))
num_cats_val = len(os.listdir(validation_cats_dir))
num_dogs_val = len(os.listdir(validation_dogs_dir))

total_train = num_cats_tr + num_dogs_tr
total_val   = num_cats_val + num_dogs_val

print('Total training cat images:',  num_cats_tr)
print('Total training dog images:',  num_dogs_tr)
print('Total validation cat images:', num_cats_val)
print('Total validation dog images:', num_dogs_val)
print('--')
print('Total training images:',   total_train)
print('Total validation images:', total_val)

In [ ]:
# ✏️ TODO — Set hyperparameters
batch_size = 128
epochs     = 15
IMG_HEIGHT = 150
IMG_WIDTH  = 150

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `os.listdir(path)` | Returns a Python list of all file names in the given directory. |
| `len(os.listdir(...))` | Counts the number of files (i.e., images) in the directory. |
| `total_train` | The total number of training images (cats + dogs). Used to calculate `steps_per_epoch`. |
| `total_val` | The total number of validation images. Used to calculate `validation_steps`. |
| `batch_size = 128` | Number of images processed in one gradient update step. Larger batches use more memory but train faster. |
| `epochs = 15` | Number of complete passes through the entire training dataset. |
| `IMG_HEIGHT / IMG_WIDTH = 150` | All images will be resized to 150 × 150 pixels so the CNN always receives identically-shaped inputs. |

✏️ **TODO**

## Step 5: Create Image Data Generators

**Objective:** Pre-process images for model training using `ImageDataGenerator`.

**Instructions:**
1. Create `train_image_generator = ImageDataGenerator(rescale=1./255)`.
2. Create `validation_image_generator = ImageDataGenerator(rescale=1./255)`.
3. Use `flow_from_directory` on the training generator (shuffle=True).
4. Use `flow_from_directory` on the validation generator (no shuffle).

In [ ]:
# ✏️ TODO — Create ImageDataGenerators for training and validation
train_image_generator      = ImageDataGenerator(rescale=1./255)
validation_image_generator = ImageDataGenerator(rescale=1./255)

In [ ]:
# ✏️ TODO — Use flow_from_directory to load images
train_data_gen = train_image_generator.flow_from_directory(
    batch_size  = batch_size,
    directory   = train_dir,
    shuffle     = True,
    target_size = (IMG_HEIGHT, IMG_WIDTH),
    class_mode  = 'binary'
)

val_data_gen = validation_image_generator.flow_from_directory(
    batch_size  = batch_size,
    directory   = validation_dir,
    target_size = (IMG_HEIGHT, IMG_WIDTH),
    class_mode  = 'binary'
)

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `ImageDataGenerator` | Generates batches of image data during training, optionally applying preprocessing and augmentation. |
| `rescale=1./255` | Normalises pixel values from `[0, 255]` to `[0, 1]`, which stabilises gradient descent. |
| `flow_from_directory()` | Reads images directly from a folder structure and automatically assigns class labels based on subdirectory names. |
| `target_size=(IMG_HEIGHT, IMG_WIDTH)` | Resizes every image to 150 × 150 pixels so all inputs share the same shape. |
| `class_mode='binary'` | Tells the generator to produce binary labels (0 or 1), suitable for two-class classification. |
| `shuffle=True` | Randomly shuffles the training images each epoch to prevent the model from learning the order of samples. |

✏️ **TODO**

## Step 6: Visualize Sample Training Images

**Objective:** Verify the data loading by plotting sample images.

**Instructions:**
1. Use `next(train_data_gen)` to fetch one batch. Unpack as `(sample_images, _)`.
2. Define a `plotImages(images_arr)` function that creates a 1×5 grid.
3. Call `plotImages(sample_images[:5])`.

In [ ]:
# ✏️ TODO — Fetch a batch of training images
sample_training_images, _ = next(train_data_gen)

In [ ]:
# ✏️ TODO — Define plotImages function and visualise 5 sample images
def plotImages(images_arr):
    fig, axes = plt.subplots(1, 5, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plotImages(sample_training_images[:5])

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `next(train_data_gen)` | Calls Python's built-in `next()` on the generator object, fetching the next batch of `(images, labels)` pairs. |
| `sample_training_images, _` | Tuple unpacking: stores the image array in `sample_training_images` and discards the labels with the underscore `_`. |
| `plt.subplots(1, 5, figsize=(20, 20))` | Creates a figure with 1 row and 5 columns of axes, each large enough to display an image. |
| `axes.flatten()` | Converts the 2-D array of axes into a 1-D list for easier iteration. |
| `zip(images_arr, axes)` | Pairs each image with its corresponding subplot axis. |
| `ax.imshow(img)` | Renders the image array on the current axis. |
| `ax.axis('off')` | Hides tick marks and axis labels for a cleaner look. |
| `plt.tight_layout()` | Automatically adjusts spacing between subplots to prevent overlapping. |

---
# Part 2 — Model Building
---

✏️ **TODO**

## Step 7: Define the CNN Architecture

**Objective:** Build a CNN for binary image classification.

**Instructions:**
1. Initialize a `Sequential` model.
2. Add three Conv2D + MaxPooling2D blocks (16, 32, 64 filters).
3. Flatten the output, add Dense(512, relu), then Dense(1).

In [ ]:
# ✏️ TODO — Build the CNN model
model = Sequential([
    Conv2D(16, 3, padding='same', activation='relu',
           input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(),

    Conv2D(32, 3, padding='same', activation='relu'),
    MaxPooling2D(),

    Conv2D(64, 3, padding='same', activation='relu'),
    MaxPooling2D(),

    Flatten(),
    Dense(512, activation='relu'),
    Dense(1)
])

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `Sequential([...])` | Creates a linear stack of layers. Each layer feeds directly into the next. |
| `Conv2D(16, 3, padding='same', activation='relu')` | A 2-D convolutional layer with 16 filters of size 3×3. `padding='same'` preserves the spatial dimensions of the output. `relu` introduces non-linearity. |
| `input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)` | Specifies the shape of a single input image: 150 × 150 pixels with 3 colour channels (RGB). Only needed on the first layer. |
| `MaxPooling2D()` | Reduces spatial dimensions by taking the maximum value in each 2×2 window; reduces computation and helps prevent overfitting. |
| `Conv2D(32, ...)` / `Conv2D(64, ...)` | Deeper layers with more filters capture increasingly complex features (edges → textures → shapes). |
| `Flatten()` | Converts the 3-D feature maps into a 1-D vector so they can be fed into fully-connected Dense layers. |
| `Dense(512, activation='relu')` | A fully-connected layer with 512 neurons. Combines spatial features to make high-level decisions. |
| `Dense(1)` | The output neuron. Its raw logit is compared against a 0.5 threshold (or passed through a sigmoid) for binary classification. |

✏️ **TODO**

## Step 8: Compile the Model

**Objective:** Configure the model for training.

**Instructions:**
1. Call `model.compile(...)` with Adam optimizer, BinaryCrossentropy loss (`from_logits=True`), and accuracy metrics.
2. Call `model.summary()` to display the network structure.

In [ ]:
# ✏️ TODO — Compile the model
model.compile(
    optimizer = 'adam',
    loss      = tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics   = ['accuracy']
)

In [ ]:
# ✏️ TODO — Print model summary
model.summary()

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `model.compile(...)` | Configures the model for training by specifying the optimiser, loss function, and evaluation metrics. |
| `optimizer='adam'` | Adam (Adaptive Moment Estimation) automatically adjusts the learning rate per parameter, making it a robust default choice. |
| `tf.keras.losses.BinaryCrossentropy(from_logits=True)` | The loss function for binary classification. `from_logits=True` tells Keras that the final layer outputs raw scores (logits), not probabilities. |
| `metrics=['accuracy']` | Tracks classification accuracy on both training and validation sets during each epoch. |
| `model.summary()` | Prints a table showing each layer, its output shape, and the number of trainable parameters. Useful for verifying the architecture before training. |

---
# Part 3 — Model Training and Evaluation
---

✏️ **TODO**

## Step 9: Train the Model

**Objective:** Fit the CNN on the training data while monitoring validation performance.

**Instructions:**
1. Call `model.fit(...)` with `train_data_gen`, `steps_per_epoch = total_train // batch_size`, `epochs`, validation data, and `validation_steps = total_val // batch_size`.
2. Store the result in `history`.

In [ ]:
# ✏️ TODO — Train the model
history = model.fit(
    train_data_gen,
    steps_per_epoch  = total_train // batch_size,
    epochs           = epochs,
    validation_data  = val_data_gen,
    validation_steps = total_val // batch_size
)

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `model.fit(...)` | Starts the training loop. Keras repeatedly calls the generator to fetch batches, computes the loss, and updates weights via backpropagation. |
| `train_data_gen` | The generator that yields batches of training images and labels. |
| `steps_per_epoch = total_train // batch_size` | Number of batches to draw per epoch. Using `//` (integer division) ensures we process all images without a partial batch. |
| `epochs = epochs` | Passes the `epochs` variable (15) telling the model how many full passes to make over the training data. |
| `validation_data = val_data_gen` | Provides the validation generator so Keras can compute validation loss and accuracy after each epoch. |
| `validation_steps = total_val // batch_size` | Number of validation batches to draw per evaluation round. |
| `history` | A `History` object returned by `fit`, storing `loss`, `accuracy`, `val_loss`, and `val_accuracy` for every epoch — used to plot learning curves. |

✏️ **TODO**

## Step 10: Visualize Training and Validation Metrics

**Objective:** Plot accuracy and loss curves to assess model performance.

**Instructions:**
1. Extract `acc`, `val_acc`, `loss`, `val_loss` from `history.history`.
2. Create `epochs_range = range(epochs)`.
3. Plot two side-by-side subplots: accuracy and loss.
4. Call `plt.show()`.

In [ ]:
# ✏️ TODO — Visualize training and validation accuracy and loss
acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))

# Training and Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Training and Validation Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

### 💡 Detailed Code Explanation
| Syntax/Method | Detailed Explanation/Purpose |
|---|---|
| `history.history['accuracy']` | Retrieves the list of training accuracy values, one per epoch, stored in the `History` object. |
| `history.history['val_accuracy']` | Retrieves the list of validation accuracy values, one per epoch. |
| `history.history['loss']` / `'val_loss'` | Retrieves training and validation loss values respectively. |
| `range(epochs)` | Creates a sequence 0, 1, 2, … 14 used as the x-axis for the plots. |
| `plt.figure(figsize=(8, 8))` | Creates a new figure with a specific size (in inches). |
| `plt.subplot(1, 2, 1)` | Selects the first subplot in a 1-row, 2-column grid. |
| `plt.plot(...)` | Draws a line chart. The first two arguments are the x and y data; `label` sets the legend entry. |
| `plt.legend(loc='lower right')` | Places the legend in the lower-right corner of the axes. |
| **Interpreting the curves** | If training accuracy is much higher than validation accuracy, the model is **overfitting**. If both are low, it may be **underfitting**. Healthy learning shows both curves converging. |